#### Final Scraping with enhanced code

In [1]:
%pip install curl_cffi beautifulsoup4 mysql-connector-python numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
from curl_cffi import requests as cffi_requests
import bs4
import mysql.connector
import numpy

print("All installed correctly")

All installed correctly


In [3]:
# import all the required libraries
from curl_cffi import requests  # Drop-in replacement that impersonates Chrome TLS
from bs4 import BeautifulSoup
import re
import mysql.connector
from datetime import date
import numpy as np
import time
import random

In [4]:
categories ={ 'Sports' : ['Badminton', 'Cycle', 'Balance Bikes', 'Exercise Bikes handpicked', 'yoga', 'camping', 'kids cycle', 'Treadmils', 'Gym Combo', 'Cricket', 'Ball Sports', 'Indoor Sports'],
              'Books' : ['Guitars','Microphones', 'Keyboards', 'cajons', 'Amplifiers', 'Children Books', 'Boxsets', 'Fiction books', 'Non Fiction books', 'school books', 'Comics'],
              'Furniture' : ['Mattresses', 'Office Chairs', 'Beds', 'ward drobes', 'office tables', 'kids furniture', 'sofa beds', 'Laptop tables', 'shoe racks', 'cofffe tables', 'Dining sets']
            }

In [5]:
def get_next_page(soup):
    next_page = soup.find('a', class_ = 's-pagination-next')
    if next_page and 'href' in next_page.attrs:
        return "https://www.amazon.in" + next_page['href']
    return None

In [7]:
%pip install -q python-dotenv

import os
import dotenv
from dotenv import load_dotenv

load_dotenv()


if __name__ == '__main__':

    # ── Rotate across multiple Chrome/Safari TLS fingerprints ──────────────────
    IMPERSONATION_PROFILES = [
        "chrome110", "chrome116", "chrome119", "chrome120",
        "safari15_5", "safari17_0",
    ]

    def get_headers():
        user_agents = [
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_2) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15",
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        ]
        return {
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
            'Accept-Language': 'en-IN,en;q=0.9,hi;q=0.8',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
            'Sec-Fetch-Dest': 'document',
            'Sec-Fetch-Mode': 'navigate',
            'Sec-Fetch-Site': 'same-origin',
            'Sec-Fetch-User': '?1',
            'DNT': '1',
            'User-Agent': random.choice(user_agents),
            'Referer': 'https://www.google.com/',   # look like organic search traffic
        }

    def is_blocked(html):
        return any(x in html.lower() for x in [
            "captcha", "robot check", "enter the characters",
            "type the characters", "sorry, we just need to make sure"
        ])

    def make_session():
        """Spin up a brand-new session with a random TLS profile."""
        profile = random.choice(IMPERSONATION_PROFILES)
        print(f"  [session] new session → impersonate={profile}")
        s = requests.Session(impersonate=profile)
        s.headers.update(get_headers())
        return s

    def warm_session(s):
        """Visit homepage + a random category page to seed cookies naturally."""
        try:
            s.get("https://www.amazon.in", timeout=15)
            time.sleep(random.uniform(2, 4))
            s.get("https://www.amazon.in/gp/bestsellers/", timeout=15)
            time.sleep(random.uniform(1, 3))
        except Exception:
            pass

    today = date.today()

    # ── MySQL Connection ────────────────────────────────────────────────────────
    conn = mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASS"),
    )
    cursor = conn.cursor()
    cursor.execute("CREATE DATABASE IF NOT EXISTS ecommerce")
    cursor.execute("USE ecommerce")
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS amazon_products (
            id INT AUTO_INCREMENT PRIMARY KEY,
            ASIN VARCHAR(20),
            title TEXT,
            price VARCHAR(50),
            rating VARCHAR(10),
            link TEXT,
            scrape_date DATE
        )
    """)

    insert_query = """
        INSERT INTO amazon_products
        (ASIN, title, price, rating, link, scrape_date)
        VALUES (%s, %s, %s, %s, %s, %s)
    """

    # ── Initial warm-up ─────────────────────────────────────────────────────────
    session = make_session()
    print("Warming up session...")
    warm_session(session)

    seen_asins = set()
    MAX_SESSION_RETRIES = 2     # how many fresh sessions to try per page before skipping

    for category, products in categories.items():
        print(f"\n===== CATEGORY: {category} =====")

        for product in products:
            print(f"\nSearching product: {product}")
            keyword = re.sub(r'\s+', '+', product)
            url = f"https://www.amazon.in/s?k={keyword}"
            page_no = 1

            while url and page_no <= 2:
                print(f"  Scraping page {page_no}")
                time.sleep(random.uniform(3, 7))     # shorter, smarter delays

                page_ok = False
                for attempt in range(MAX_SESSION_RETRIES):
                    try:
                        response = session.get(url, timeout=20, headers=get_headers())
                    except Exception as e:
                        print(f"  ❌ Request error (attempt {attempt+1}): {e}")
                        session = make_session()
                        warm_session(session)
                        time.sleep(random.uniform(4, 8))
                        continue

                    if response.status_code == 200 and not is_blocked(response.text):
                        page_ok = True
                        break

                    # ── blocked → spawn a completely fresh session ──────────────
                    print(f"  ⚠ Blocked (attempt {attempt+1}) → spawning fresh session...")
                    session = make_session()
                    warm_session(session)
                    time.sleep(random.uniform(5, 10))

                if not page_ok:
                    print(f"  ✖ Could not fetch page {page_no} for '{product}' after {MAX_SESSION_RETRIES} attempts → skipping")
                    url = None
                    break

                soup = BeautifulSoup(response.content, "html.parser")

                count = 0
                for item in soup.select("div.s-result-item[data-asin]"):
                    asin = item.get("data-asin")
                    if not asin or asin in seen_asins:
                        continue
                    seen_asins.add(asin)

                    title_el = item.select_one("h2 span")
                    price_el = item.select_one("span.a-price span.a-offscreen")
                    rating_el = item.select_one("span.a-icon-alt")

                    title  = title_el.text.strip()        if title_el  else None
                    price  = price_el.text.strip()        if price_el  else None
                    rating = rating_el.text.split()[0]    if rating_el else None
                    link   = "https://www.amazon.in/dp/" + asin

                    if not title or not link:
                        continue

                    cursor.execute(insert_query, (asin, title, price, rating, link, today))
                    count += 1

                conn.commit()
                print(f"  ✅ Inserted {count} products")

                next_btn = soup.select_one("a.s-pagination-next")
                url = "https://www.amazon.in" + next_btn["href"] if next_btn else None
                page_no += 1

    cursor.close()
    conn.close()
    print("\n✅ All categories & products scraped successfully")


Note: you may need to restart the kernel to use updated packages.
  [session] new session → impersonate=chrome116
Warming up session...

===== CATEGORY: Sports =====

Searching product: Badminton
  Scraping page 1
  ✅ Inserted 59 products
  Scraping page 2


KeyboardInterrupt: 

In [ ]:
random.uniform(6,10)

7.856276068158715